In [333]:
import pandas as pd
import geopandas as gpd
from census import Census
from us import states
from ipumspy import (
    IpumsApiClient,
    AggregateDataExtract,
    NhgisDataset,
)
from pathlib import Path
import glob
import zipfile
pd.set_option('display.max_columns', None) 
pd.set_option('display.max_rows', 50) 

Loading General Data

In [334]:
p1_population_columns_2010 = {
    "P001003": "WHITE",      # White alone
    "P001004": "BLACK",      # Black or African American alone
    "P001005": "AMIN",       # American Indian and Alaska Native alone
    "P001006": "ASIAN",      # Asian alone
    "P001007": "NHPI",       # Native Hawaiian and Other Pacific Islander alone
    "P001008": "OTHER",  # Some Other Race alone
    "P001009": "2MORE",  # Two or more races
    "P001001": "TOTPOP" #Total Population
               
}

p1_population_columns_2020 = {
    "P1_003N": "WHITE",         # White alone
    "P1_004N": "BLACK",         # Black or African American alone
    "P1_005N": "AMIN",          # American Indian and Alaska Native alone
    "P1_006N": "ASIAN",         # Asian alone
    "P1_007N": "NHPI",          # Native Hawaiian and Other Pacific Islander alone
    "P1_008N": "OTHER",     # Some Other Race alone
    "P1_009N": "2MORE",     # Two or more races
    "P1_001N" : "TOTPOP"  #Total Population        
}

In [335]:
FIPS = "17,18,55"

In [336]:
CENSUS_KEY = "7e1b79ce2adac634987a423b6d7fb99510fee50e"
IPUMS_KEY = "59cba10d8a5da536fc06b59ddf0a32fc7c744e15949a44bb2ffe21fe"

In [337]:
census_2010 = Census(
    key=CENSUS_KEY,      # We use the provided Census API key.
    year=2010    # We specify that we would like to use the 2020 Census data.
)

In [338]:
census_2020 = Census(
    key=CENSUS_KEY,      # We use the provided Census API key.
    year=2020    # We specify that we would like to use the 2020 Census data.
)

In [339]:
ipums = IpumsApiClient(IPUMS_KEY)

In [340]:
# Illinois 2020 block groups
il_bg_2020 = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2020/BG/tl_2020_17_bg.zip"
)




# Illinois 2010 block groups
il_bg_2010 = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2010/BG/2010/tl_2010_17_bg10.zip"
)



Tracts

In [341]:
# Illinois 2020 tracts
il_tracts_2020 = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2020/TRACT/tl_2020_17_tract.zip"
)

# Illinois 2010 tracts
il_tracts_2010 = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2010/TRACT/2010/tl_2010_17_tract10.zip"
)

# Indiana 2020 tracts
in_tracts_2020 = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2020/TRACT/tl_2020_18_tract.zip"
)

# Indiana 2010 tracts
in_tracts_2010 = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2010/TRACT/2010/tl_2010_18_tract10.zip"
)

# Wisconsin 2020 tracts
wi_tracts_2020 = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2020/TRACT/tl_2020_55_tract.zip"
)

# Wisconsin 2010 tracts
wi_tracts_2010 = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2010/TRACT/2010/tl_2010_55_tract10.zip"
)

In [342]:
shapefile_tracts_2020 = gpd.GeoDataFrame(
    pd.concat([il_tracts_2020, in_tracts_2020, wi_tracts_2020], ignore_index=True), 
    crs=il_tracts_2020.crs)

shapefile_tracts_2020.columns = shapefile_tracts_2020.columns.astype(str)

shapefile_tracts_2010 = gpd.GeoDataFrame(
    pd.concat([il_tracts_2010, in_tracts_2010, wi_tracts_2010], ignore_index=True), 
    crs=il_tracts_2010.crs)

shapefile_tracts_2010.columns = shapefile_tracts_2020.columns.astype(str)

In [343]:
df_tracts_2010

,name,WHITE,BLACK,AMIN,ASIAN,NHPI,OTHER,2MORE,TOTPOP,state,county,tract
GEOID,,,,,,,,,,,,
17001000100,"Census Tract 1, Adams County, Illinois",4412.0,62.0,7.0,45.0,1.0,24.0,76.0,4627,17,001,000100
17001000201,"Census Tract 2.01, Adams County, Illinois",1882.0,53.0,2.0,9.0,1.0,3.0,36.0,1986,17,001,000201
17001000202,"Census Tract 2.02, Adams County, Illinois",2688.0,215.0,9.0,14.0,1.0,21.0,51.0,2999,17,001,000202
17001000400,"Census Tract 4, Adams County, Illinois",3527.0,582.0,9.0,13.0,0.0,15.0,176.0,4322,17,001,000400
17001000500,"Census Tract 5, Adams County, Illinois",1872.0,380.0,5.0,6.0,1.0,3.0,70.0,2337,17,001,000500
...,...,...,...,...,...,...,...,...,...,...,...,...
55141011300,"Census Tract 113, Wood County, Wisconsin",4265.0,37.0,40.0,171.0,2.0,47.0,45.0,4607,55,141,011300
55141011400,"Census Tract 114, Wood County, Wisconsin",4941.0,20.0,40.0,189.0,2.0,41.0,75.0,5308,55,141,011400
55141011500,"Census Tract 115, Wood County, Wisconsin",5776.0,20.0,32.0,56.0,1.0,29.0,31.0,5945,55,141,011500


In [344]:
df_tracts_2010 = census_2010.pl.get(
    ("NAME", *p1_population_columns_2010),
    geo={
        "for": "tract:*",
        "in": f"state:{FIPS} county:*",
    }, 
)

df_tracts_2010 = pd.DataFrame(df_tracts_2010).rename(
    columns={"NAME": "name", **p1_population_columns_2010}
)

In [345]:
df_tracts_2020 = census_2020.pl.get(
    ("NAME", *p1_population_columns_2020),
    geo={
        "for": "tract:*",
        "in": f"state:{FIPS} county:*",
    }, 
)

df_tracts_2020 = pd.DataFrame(df_tracts_2020).rename(
    columns={"NAME": "name", **p1_population_columns_2020}
)

In [346]:
shapefile_tracts_2020["GEOID"] = (
    shapefile_tracts_2020["STATEFP"]
    + shapefile_tracts_2020["COUNTYFP"]
    + shapefile_tracts_2020["TRACTCE"]
)
shapefile_tracts_2020 = shapefile_tracts_2020.set_index("GEOID")

shapefile_tracts_2010["GEOID"] = (
    shapefile_tracts_2010["STATEFP"]
    + shapefile_tracts_2010["COUNTYFP"]
    + shapefile_tracts_2010["TRACTCE"]
)
shapefile_tracts_2010 = shapefile_tracts_2010.set_index("GEOID")

df_tracts_2010["GEOID"] = (
    df_tracts_2010["state"]
    + df_tracts_2010["county"]
    + df_tracts_2010["tract"]
)

df_tracts_2010 = df_tracts_2010.set_index("GEOID")

df_tracts_2020["GEOID"] = (
    df_tracts_2020["state"]
    + df_tracts_2020["county"]
    + df_tracts_2020["tract"]
)


df_tracts_2020 = df_tracts_2020.set_index("GEOID")

In [347]:
merged_gdf = shapefile_tracts_2020.merge(
    df_tracts_2020, left_on="GEOID", right_on="GEOID", suffixes=("", "_df")
    )

merged_gdf["POC"] = merged_gdf["TOTPOP"] - merged_gdf["WHITE"].astype(int)


merged_gdf.to_file("Chicago_Maup_Data/tracts/tracts_joined_2020")

/Users/samstephenson/venvs/py314/lib/python3.14/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'name' to 'name_1'
  ogr_write(


In [348]:
merged_gdf = shapefile_tracts_2010.merge(
    df_tracts_2010, left_on="GEOID", right_on="GEOID", suffixes=("", "_df")
    )

merged_gdf["POC"] = merged_gdf["TOTPOP"].astype(int) - merged_gdf["WHITE"].astype(int)


merged_gdf.to_file("Chicago_Maup_Data/tracts/tracts_joined_2010")

/Users/samstephenson/venvs/py314/lib/python3.14/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'name' to 'name_1'
  ogr_write(


Block Groups

In [349]:
# Illinois (FIPS 17)
il_bg_2020 = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2020/BG/tl_2020_17_bg.zip"
)

# Indiana (FIPS 18)
in_bg_2020 = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2020/BG/tl_2020_18_bg.zip"
)

# Wisconsin (FIPS 55)
wi_bg_2020 = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2020/BG/tl_2020_55_bg.zip"
)

# Illinois (FIPS 17)
il_bg_2010 = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2010/BG/2010/tl_2010_17_bg10.zip"
)

# Indiana (FIPS 18)
in_bg_2010 = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2010/BG/2010/tl_2010_18_bg10.zip"
)

# Wisconsin (FIPS 55)
wi_bg_2010 = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2010/BG/2010/tl_2010_55_bg10.zip"
)

In [350]:
shapefile_bg_2020 = gpd.GeoDataFrame(
    pd.concat([il_bg_2020, in_bg_2020, wi_bg_2020], ignore_index=True), 
    crs=il_tracts_2020.crs)

shapefile_bg_2020.columns = shapefile_bg_2020.columns.astype(str)

shapefile_bg_2010 = gpd.GeoDataFrame(
    pd.concat([il_bg_2010, in_bg_2010, wi_bg_2010], ignore_index=True), 
    crs=il_bg_2010.crs)

shapefile_bg_2010.columns = shapefile_bg_2020.columns.astype(str)

In [351]:
df_bg_2010 = census_2010.pl.get(
    ("NAME", *p1_population_columns_2010),
    geo={
        "for": "block group:*",
        "in": f"state:{FIPS} county:* tract:*",
    },
)

df_bg_2010 = pd.DataFrame(df_bg_2010).rename(
    columns={"NAME": "name", **p1_population_columns_2010}
)

df_bg_2020 = census_2020.pl.get(
    ("NAME", *p1_population_columns_2020),
    geo={
        "for": "block group:*",
        "in": f"state:{FIPS} county:* tract:*",
    },
)

df_bg_2020 = pd.DataFrame(df_bg_2020).rename(
    columns={"NAME": "name", **p1_population_columns_2020}
)

In [352]:
shapefile_bg_2020["GEOID"] = (
    shapefile_bg_2020["STATEFP"]
    + shapefile_bg_2020["COUNTYFP"]
    + shapefile_bg_2020["TRACTCE"]
    + shapefile_bg_2020["BLKGRPCE"]
)
shapefile_bg_2020 = shapefile_bg_2020.set_index("GEOID")

shapefile_bg_2010["GEOID"] = (
    shapefile_bg_2010["STATEFP"]
    + shapefile_bg_2010["COUNTYFP"]
    + shapefile_bg_2010["TRACTCE"]
    + shapefile_bg_2010["BLKGRPCE"]
)
shapefile_bg_2010 = shapefile_bg_2010.set_index("GEOID")

df_bg_2010["GEOID"] = (
    df_bg_2010["state"]
    + df_bg_2010["county"]
    + df_bg_2010["tract"]
    + df_bg_2010["block group"]
)

df_bg_2010 = df_bg_2010.set_index("GEOID")

df_bg_2020["GEOID"] = (
    df_bg_2020["state"]
    + df_bg_2020["county"]
    + df_bg_2020["tract"]
    + df_bg_2020["block group"]
)


df_bg_2020 = df_bg_2020.set_index("GEOID")

In [353]:
merged_gdf = shapefile_bg_2010.merge(
    df_bg_2010, left_on="GEOID", right_on="GEOID", suffixes=("", "_df")
    )

merged_gdf["POC"] = merged_gdf["TOTPOP"].astype(int) - merged_gdf["WHITE"].astype(int)


merged_gdf.to_file("Chicago_Maup_Data/block groups/bg_joined_2010")

/var/folders/t8/lj11mg1j3x76rrnc0m3pjp5h0000gn/T/ipykernel_41815/1409199496.py:8: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  merged_gdf.to_file("Chicago_Maup_Data/block groups/bg_joined_2010")
/Users/samstephenson/venvs/py314/lib/python3.14/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'block group' to 'block grou'
  ogr_write(


In [354]:
merged_gdf = shapefile_bg_2020.merge(
    df_bg_2020, left_on="GEOID", right_on="GEOID", suffixes=("", "_df")
    )

merged_gdf["POC"] = merged_gdf["TOTPOP"].astype(int) - merged_gdf["WHITE"].astype(int)


merged_gdf.to_file("Chicago_Maup_Data/block groups/bg_joined_2020")

/var/folders/t8/lj11mg1j3x76rrnc0m3pjp5h0000gn/T/ipykernel_41815/3063509967.py:8: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  merged_gdf.to_file("Chicago_Maup_Data/block groups/bg_joined_2020")
/Users/samstephenson/venvs/py314/lib/python3.14/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'block group' to 'block grou'
  ogr_write(


Blocks

In [355]:
# Illinois 2020 blocks
il_blocks_2020 = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2020PL/STATE/17_ILLINOIS/17/tl_2020_17_tabblock20.zip"
)

# Wisconsin 2020 blocks
wi_blocks_2020 = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2020PL/STATE/55_WISCONSIN/55/tl_2020_55_tabblock20.zip"
)

# Wisconsin 2010 blocks
wi_blocks_2010 = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2010/TABBLOCK/2010/tl_2010_55_tabblock10.zip"
)

# Indiana 2020 blocks
in_blocks_2020 = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2020PL/STATE/18_INDIANA/18/tl_2020_18_tabblock20.zip"
)

# Indiana 2010 blocks
in_blocks_2010 = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2010/TABBLOCK/2010/tl_2010_18_tabblock10.zip"
)

in_blocks_2020 = in_blocks_2020.to_crs(il_blocks_2020.crs)
wi_blocks_2020 = wi_blocks_2020.to_crs(il_blocks_2020.crs)


in_blocks_2010 = in_blocks_2010.to_crs(il_blocks_2010.crs)
wi_blocks_2010 = wi_blocks_2010.to_crs(il_blocks_2010.crs)


shapefile_blocks_2020 = gpd.GeoDataFrame(
    pd.concat([il_blocks_2020, in_blocks_2020, wi_blocks_2020], ignore_index=True), 
    crs=il_blocks_2020.crs)

shapefile_blocks_2010 = gpd.GeoDataFrame(
    pd.concat([il_blocks_2010, in_blocks_2010, wi_blocks_2010], ignore_index=True), 
    crs=il_blocks_2010.crs)

In [356]:
shapefile_blocks_2020["GEOID"] = (
    shapefile_blocks_2020["STATEFP20"]
    + shapefile_blocks_2020["COUNTYFP20"]
    + shapefile_blocks_2020["TRACTCE20"]
    + shapefile_blocks_2020["BLOCKCE20"]
)
shapefile_blocks_2020 = shapefile_blocks_2020.set_index("GEOID")



In [357]:
df_blocks_2020 = census_2020.pl.get(
    ("NAME", *p1_population_columns_2020),
    geo={
        "for": "block:*",
        "in": f"state: {FIPS} county:* tract:*",
    }, 
)

df_blocks_2020 = pd.DataFrame(df_blocks_2020).rename(
    columns={"NAME": "name", **p1_population_columns_2020}
)

In [358]:
df_blocks_2020
df_blocks_2020["GEOID"] = (
    df_blocks_2020["state"]
    + df_blocks_2020["county"]
    + df_blocks_2020["tract"]
    + df_blocks_2020["block"]
)
df_blocks_2020 = df_blocks_2020.set_index("GEOID")

In [359]:
merged_gdf = shapefile_blocks_2020.merge(
    df_blocks_2020, left_on="GEOID", right_on="GEOID", suffixes=("", "_df")
    )

merged_gdf["POC"] = merged_gdf["TOTPOP"] - merged_gdf["WHITE"].astype(int)


merged_gdf.to_file("Chicago_Maup_Data/blocks/blocks_joined_2020")

2010

In [360]:
extract = AggregateDataExtract( #reading in csv block data from nhgis (blocks not supported for 2010 by the census api)
    datasets=[
        NhgisDataset(
            name="2010_SF1a",
            data_tables=[
                "P5"
            ],
            geog_levels=["block"]
        )
    ],
    collection="nhgis"
)

In [361]:
# Submit the extract request
extract = ipums.submit_extract(extract)

# Wait until it's finished
ipums.wait_for_extract(extract)

out_dir = Path("Chicago_Maup_Data/blocks")

# Download the completed extract
ipums.download_extract(extract, download_dir=out_dir)

In [362]:
zip_path = next(out_dir.glob("*.zip"))

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(out_dir)

In [363]:
csv_path = next(out_dir.rglob("*.csv"))

In [364]:
df_blocks_2010 = df = pd.read_csv(csv_path)

/var/folders/t8/lj11mg1j3x76rrnc0m3pjp5h0000gn/T/ipykernel_41815/3659269922.py:1: DtypeWarning: Columns (0: AIANHHCC, 1: TTRACTA, 2: TBLKGRPA, 3: UATYPE, 4: SLDUA, 5: SLDLA, 6: VTD, 7: VTDI) have mixed types. Specify dtype option on import or set low_memory=False.
  df_blocks_2010 = df = pd.read_csv(csv_path)


In [365]:
df_blocks_2010["GEOID"] = (
    df_blocks_2010["STATEA"].astype(str).str.zfill(2)
    + df_blocks_2010["COUNTYA"].astype(str).str.zfill(3)
    + df_blocks_2010["TRACTA"].astype(str).str.zfill(6)
    + df_blocks_2010["BLOCKA"].astype(str).str.zfill(4)
)

In [366]:
shapefile_blocks_2010["GEOID"] = (
    shapefile_blocks_2010["STATEFP10"]
    + shapefile_blocks_2010["COUNTYFP10"]
    + shapefile_blocks_2010["TRACTCE10"]
    + shapefile_blocks_2010["BLOCKCE10"]
)
shapefile_blocks_2010 = shapefile_blocks_2010.set_index("GEOID")

In [367]:
NHGIS_MAPPINGS = {"H7Z001": "TOTPOP",
                  "H7Z003": "WHITE", #white nonhispanic
                  "H7Z004": "BLACK" #black nonhispanic
                  }

In [368]:
df_blocks_2010 = df_blocks_2010.rename(
    {k: v for k, v in NHGIS_MAPPINGS.items()}, axis=1
    )

In [369]:
merged_gdf = shapefile_blocks_2010.merge(
    df_blocks_2010, left_on="GEOID", right_on="GEOID", suffixes=("", "_df")
    )


merged_gdf["POC"] = merged_gdf["TOTPOP"] - merged_gdf["WHITE"].astype(int)

merged_gdf.to_file("Chicago_Maup_Data/blocks/blocks_joined_2010")

/Users/samstephenson/venvs/py314/lib/python3.14/site-packages/pyogrio/raw.py:733: RuntimeWarning: 2GB file size limit reached for Chicago_Maup_Data/blocks/blocks_joined_2010/blocks_joined_2010.dbf. Going on, but might cause compatibility issues with third party software
  ogr_write(


/var/folders/t8/lj11mg1j3x76rrnc0m3pjp5h0000gn/T/ipykernel_60159/1303932211.py:2: DtypeWarning: Columns (0: AIANHHCC, 1: TTRACTA, 2: TBLKGRPA, 3: CD116A, 4: SLDU18A, 5: SLDL18A, 6: VTD, 7: VTDI) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("Chicago_2020_Maup_Data/blocks/nhgis0010_csv_block/nhgis0010_ds258_2020_block.csv", encoding_errors="replace")